In [4]:
import numpy as np
import random
import math
import copy
from typing import List, Tuple, Dict
import time

# ============================================================================
# CLASSE VRP ET CHARGEMENT DES DONNÉES
# ============================================================================
import math
import numpy as np

class VRPInstance:
    def __init__(self):
        self.n_customers = 0
        self.n_vehicles = 0
        self.capacity = 0
        self.coords = []
        self.demands = []
        self.distances = None

    def calculate_distances(self):
        n = len(self.coords)
        self.distances = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                dx = self.coords[i][0] - self.coords[j][0]
                dy = self.coords[i][1] - self.coords[j][1]
                self.distances[i][j] = math.sqrt(dx*dx + dy*dy)

    def load_from_file(self, filename: str):
        """Charge le dataset depuis un fichier"""
        with open(filename, 'r') as f:
            lines = f.readlines()

        # Parser le fichier
        reading_coords = False
        reading_demands = False
        coords_list = []
        demands_list = []
        dimension = 0

        for line in lines:
            line = line.strip()
            if not line:
                continue

            if 'DIMENSION' in line:
                parts = line.split(':' if ':' in line else ' ')
                dimension = int(parts[-1].strip())
                print(f"  DIMENSION trouvée: {dimension}")
            elif 'CAPACITY' in line:
                parts = line.split(':' if ':' in line else ' ')
                self.capacity = int(parts[-1].strip())
                print(f"  CAPACITY trouvée: {self.capacity}")
            elif 'VEHICLES' in line or line.startswith('VEHICLES'):
                parts = line.split(':' if ':' in line else ' ')
                self.n_vehicles = int(parts[-1].strip())
                print(f"  VEHICLES trouvé: {self.n_vehicles}")
            elif 'NODE_COORD_SECTION' in line:
                reading_coords = True
                reading_demands = False
                print("  Lecture des coordonnées...")
                continue
            elif 'DEMAND_SECTION' in line:
                reading_coords = False
                reading_demands = True
                print("  Lecture des demandes...")
                continue
            elif 'DEPOT_SECTION' in line or line == 'EOF' or line == '-1':
                reading_coords = False
                reading_demands = False
                continue

            # Lire coordonnées
            if reading_coords:
                parts = line.split()
                if len(parts) >= 3:
                    try:
                        node_id = int(parts[0])
                        x = float(parts[1])
                        y = float(parts[2])
                        coords_list.append((x, y))
                    except:
                        continue

            # Lire demandes
            if reading_demands:
                parts = line.split()
                if len(parts) >= 2:
                    try:
                        node_id = int(parts[0])
                        demand = int(parts[1])
                        demands_list.append(demand)
                    except:
                        continue

        # Vérifications
        if not coords_list:
            raise ValueError("Aucune coordonnée trouvée dans le fichier!")
        if not demands_list:
            raise ValueError("Aucune demande trouvée dans le fichier!")

        self.coords = coords_list
        self.demands = demands_list
        self.n_customers = len(coords_list) - 1  # -1 pour exclure le dépôt

        print(f"  Coordonnées chargées: {len(coords_list)} nœuds")
        print(f"  Demandes chargées: {len(demands_list)} nœuds")

        # Si n_vehicles non spécifié, estimer
        if self.n_vehicles == 0:
            if self.capacity > 0:
                total_demand = sum(self.demands[1:])
                self.n_vehicles = max(1, math.ceil(total_demand / self.capacity))
                print(f"  Nombre de véhicules estimé: {self.n_vehicles}")
            else:
                self.n_vehicles = 7  # Valeur par défaut pour F-n135-k7

        # Si capacité non spécifiée
        if self.capacity == 0:
            self.capacity = 2210  # Valeur par défaut pour F-n135-k7

        self.calculate_distances()

        print(f"\n✓ Fichier chargé avec succès!")
        print(f"  → {self.n_customers} clients")
        print(f"  → {self.n_vehicles} véhicules")
        print(f"  → Capacité: {self.capacity}")

    def load_f135_k7_embedded(self):
        """Charge le dataset F-n135-k7 intégré (backup)"""
        print("\n⚠ Utilisation des données intégrées F-n135-k7")
        self.n_customers = 134
        self.n_vehicles = 7
        self.capacity = 2210

        # Coordonnées (dépôt + 134 clients)
        coords_data = [
            (41, 49), (35, 17), (55, 45), (55, 20), (15, 30), (25, 30), (20, 50),
            (10, 43), (55, 60), (30, 60), (20, 65), (50, 35), (30, 25), (15, 10),
            (30, 5), (10, 20), (5, 30), (20, 40), (15, 60), (45, 65), (45, 20),
            (45, 10), (55, 5), (65, 35), (65, 20), (45, 30), (35, 40), (41, 37),
            (64, 42), (40, 60), (31, 52), (35, 69), (53, 52), (65, 55), (63, 65),
            (2, 60), (20, 20), (5, 5), (60, 12), (40, 25), (42, 7), (24, 12),
            (23, 3), (11, 14), (6, 38), (2, 48), (8, 56), (13, 52), (6, 68),
            (47, 47), (49, 58), (27, 43), (37, 31), (57, 29), (63, 23), (53, 12),
            (32, 12), (36, 26), (21, 24), (17, 34), (12, 24), (24, 58), (27, 69),
            (15, 77), (62, 77), (49, 73), (67, 5), (56, 39), (37, 47), (37, 56),
            (57, 68), (47, 16), (44, 17), (46, 13), (49, 11), (49, 42), (53, 43),
            (61, 52), (57, 48), (56, 37), (55, 54), (15, 47), (14, 37), (11, 31),
            (16, 22), (4, 18), (28, 18), (26, 52), (26, 35), (31, 67), (15, 19),
            (22, 22), (18, 24), (26, 27), (25, 24), (22, 27), (25, 21), (19, 21),
            (20, 26), (18, 18), (35, 35), (40, 45), (38, 42), (30, 45), (28, 40),
            (26, 38), (30, 35), (28, 35), (26, 32), (33, 41), (31, 41), (31, 38),
            (33, 38), (35, 38), (33, 35), (35, 32), (33, 32), (32, 35), (32, 41),
            (28, 41), (26, 41), (24, 38), (24, 35), (24, 32), (21, 35), (21, 32),
            (21, 29), (19, 32), (19, 29), (23, 29), (23, 26)
        ]

        # Demandes (0 pour dépôt, puis 134 clients)
        demands_data = [
            0, 19, 30, 16, 23, 11, 31, 15, 28, 8, 8, 7, 14, 6, 19, 11, 12, 26,
            17, 6, 15, 14, 7, 27, 13, 11, 16, 10, 5, 25, 17, 18, 10, 9, 11, 18,
            29, 3, 6, 17, 16, 16, 9, 21, 27, 23, 11, 14, 8, 5, 8, 16, 31, 9,
            5, 5, 7, 18, 16, 1, 27, 36, 30, 13, 10, 9, 14, 18, 2, 6, 7, 18,
            28, 3, 13, 19, 10, 9, 20, 25, 25, 36, 6, 5, 15, 25, 9, 8, 18, 13,
            14, 3, 23, 6, 26, 16, 11, 7, 41, 35, 26, 9, 15, 3, 1, 2, 22, 27,
            20, 11, 12, 10, 9, 17, 11, 18, 2, 29, 3, 6, 2, 7, 1, 15, 14, 6,
            19, 11, 28, 7, 15, 13, 9, 9, 3
        ]

        self.coords = coords_data
        self.demands = demands_data
        self.calculate_distances()

        print(f"  → {self.n_customers} clients")
        print(f"  → {self.n_vehicles} véhicules")
        print(f"  → Capacité: {self.capacity}")


               
            
           
                   
        
      
# ============================================================================
# FONCTIONS UTILITAIRES
# ============================================================================

def calculate_route_cost(route: List[int], distances: np.ndarray) -> float:
    if len(route) <= 1:
        return 0.0
    cost = distances[0][route[0]]
    for i in range(len(route) - 1):
        cost += distances[route[i]][route[i+1]]
    cost += distances[route[-1]][0]
    return cost

def calculate_total_cost(solution: List[List[int]], distances: np.ndarray) -> float:
    return sum(calculate_route_cost(route, distances) for route in solution)

def is_feasible(route: List[int], demands: List[int], capacity: int) -> bool:
    return sum(demands[c] for c in route) <= capacity

def copy_solution(solution: List[List[int]]) -> List[List[int]]:
    return [route[:] for route in solution]

# ============================================================================
# 1. NEAREST NEIGHBOR AMÉLIORÉ
# ============================================================================

def nearest_neighbor_improved(vrp: VRPInstance) -> List[List[int]]:
    print("  → Nearest Neighbor Amélioré...")
    unvisited = set(range(1, vrp.n_customers + 1))
    solution = []
    
    for v in range(vrp.n_vehicles):
        route = []
        load = 0
        current = 0
        
        while unvisited:
            # Trouver les 5 plus proches voisins
            candidates = []
            for node in unvisited:
                if load + vrp.demands[node] <= vrp.capacity:
                    dist = vrp.distances[current][node]
                    savings = vrp.distances[0][node] - dist
                    score = dist - 0.3 * savings
                    candidates.append((score, node))
            
            if not candidates:
                break
            
            candidates.sort()
            best = candidates[0][1]
            
            route.append(best)
            load += vrp.demands[best]
            current = best
            unvisited.remove(best)
        
        if route:
            solution.append(route)
    for i in range(len(solution)):
        if len(solution[i]) > 2: 
            solution[i] = two_opt(solution[i], vrp.distances)
    
    return solution

# ============================================================================
# 2. CLARKE-WRIGHT AMÉLIORÉ
# ============================================================================

def clarke_wright_improved(vrp: VRPInstance) -> List[List[int]]:
    print("  → Clarke-Wright Amélioré...")
    savings = []
    
    for i in range(1, vrp.n_customers + 1):
        for j in range(i + 1, vrp.n_customers + 1):
            s = vrp.distances[0][i] + vrp.distances[0][j] - vrp.distances[i][j]
            savings.append((s, i, j))
    
    savings.sort(reverse=True)
    
    routes = [[i] for i in range(1, vrp.n_customers + 1)]
    route_loads = [vrp.demands[i] for i in range(1, vrp.n_customers + 1)]
    node_to_route = {i: i-1 for i in range(1, vrp.n_customers + 1)}
    
    for s, i, j in savings:
        ri = node_to_route[i]
        rj = node_to_route[j]
        
        if ri != rj:
            route_i = routes[ri]
            route_j = routes[rj]
            
            if route_loads[ri] + route_loads[rj] <= vrp.capacity:
                if route_i[-1] == i and route_j[0] == j:
                    new_route = route_i + route_j
                elif route_i[0] == i and route_j[-1] == j:
                    new_route = route_j + route_i
                elif route_i[-1] == i and route_j[-1] == j:
                    new_route = route_i + route_j[::-1]
                elif route_i[0] == i and route_j[0] == j:
                    new_route = route_i[::-1] + route_j
                else:
                    continue
                
                routes[ri] = new_route
                route_loads[ri] += route_loads[rj]
                routes[rj] = []
                route_loads[rj] = 0
                
                for node in new_route:
                    node_to_route[node] = ri
    final_routes = [r for r in routes if r]    
    for i in range(len(final_routes)):
        if len(final_routes[i]) > 2:
            final_routes[i] = two_opt(final_routes[i], vrp.distances)
    return final_routes 
    
   

# ============================================================================
# 3. SWEEP ALGORITHM
# ============================================================================

def sweep_algorithm(vrp: VRPInstance) -> List[List[int]]:
    print("  → Sweep Algorithm...")
    depot = vrp.coords[0]
    
    angles = []
    for i in range(1, vrp.n_customers + 1):
        dx = vrp.coords[i][0] - depot[0]
        dy = vrp.coords[i][1] - depot[1]
        angle = math.atan2(dy, dx)
        angles.append((angle, i))
    
    angles.sort()
    sorted_customers = [c for _, c in angles]
    
    solution = []
    route = []
    load = 0
    
    for customer in sorted_customers:
        if load + vrp.demands[customer] <= vrp.capacity:
            route.append(customer)
            load += vrp.demands[customer]
        else:
            if route:
                solution.append(route)
            route = [customer]
            load = vrp.demands[customer]
    
    if route:
        solution.append(route)
    
    return solution

# ============================================================================
# 4. OPÉRATEURS D'AMÉLIORATION
# ============================================================================

def two_opt(route: List[int], distances: np.ndarray) -> List[int]:
    best = route[:]
    improved = True
    
    while improved:
        improved = False
        best_cost = calculate_route_cost(best, distances)
        
        for i in range(len(best) - 1):
            for j in range(i + 2, len(best)):
                new_route = best[:i+1] + best[i+1:j+1][::-1] + best[j+1:]
                new_cost = calculate_route_cost(new_route, distances)
                
                if new_cost < best_cost:
                    best = new_route
                    best_cost = new_cost
                    improved = True
                    break
            if improved:
                break
    
    return best

def cross_exchange(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    best_sol = copy_solution(sol)
    best_cost = calculate_total_cost(best_sol, vrp.distances)
    
    for i in range(len(sol)):
        for j in range(i + 1, len(sol)):
            for size1 in range(1, min(4, len(sol[i]) + 1)):
                for size2 in range(1, min(4, len(sol[j]) + 1)):
                    for pos1 in range(len(sol[i]) - size1 + 1):
                        for pos2 in range(len(sol[j]) - size2 + 1):
                            new_sol = copy_solution(sol)
                            
                            seg1 = new_sol[i][pos1:pos1+size1]
                            seg2 = new_sol[j][pos2:pos2+size2]
                            
                            new_sol[i] = new_sol[i][:pos1] + seg2 + new_sol[i][pos1+size1:]
                            new_sol[j] = new_sol[j][:pos2] + seg1 + new_sol[j][pos2+size2:]
                            
                            if (is_feasible(new_sol[i], vrp.demands, vrp.capacity) and
                                is_feasible(new_sol[j], vrp.demands, vrp.capacity)):
                                cost = calculate_total_cost(new_sol, vrp.distances)
                                if cost < best_cost:
                                    best_sol = new_sol
                                    best_cost = cost
    
    return best_sol

def ejection_chain(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    best_sol = copy_solution(sol)
    best_cost = calculate_total_cost(best_sol, vrp.distances)
    
    for i in range(len(sol)):
        if len(sol[i]) < 2:
            continue
        
        for pos in range(len(sol[i])):
            customer = sol[i][pos]
            
            for j in range(len(sol)):
                if i == j:
                    continue
                
                new_sol = copy_solution(sol)
                new_sol[i].pop(pos)
                
                best_insert_pos = 0
                best_insert_cost = float('inf')
                
                for insert_pos in range(len(new_sol[j]) + 1):
                    test_route = new_sol[j][:insert_pos] + [customer] + new_sol[j][insert_pos:]
                    if is_feasible(test_route, vrp.demands, vrp.capacity):
                        cost = calculate_route_cost(test_route, vrp.distances)
                        if cost < best_insert_cost:
                            best_insert_cost = cost
                            best_insert_pos = insert_pos
                
                new_sol[j].insert(best_insert_pos, customer)
                
                if is_feasible(new_sol[j], vrp.demands, vrp.capacity):
                    cost = calculate_total_cost(new_sol, vrp.distances)
                    if cost < best_cost:
                        best_sol = new_sol
                        best_cost = cost
    
    return best_sol

def string_relocation(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    best_sol = copy_solution(sol)
    best_cost = calculate_total_cost(best_sol, vrp.distances)
    
    for i in range(len(sol)):
        for string_len in range(1, min(4, len(sol[i]) + 1)):
            for pos in range(len(sol[i]) - string_len + 1):
                string = sol[i][pos:pos+string_len]
                
                for j in range(len(sol)):
                    if i == j:
                        continue
                    
                    for insert_pos in range(len(sol[j]) + 1):
                        new_sol = copy_solution(sol)
                        new_sol[i] = new_sol[i][:pos] + new_sol[i][pos+string_len:]
                        new_sol[j] = new_sol[j][:insert_pos] + string + new_sol[j][insert_pos:]
                        
                        if is_feasible(new_sol[j], vrp.demands, vrp.capacity):
                            cost = calculate_total_cost(new_sol, vrp.distances)
                            if cost < best_cost:
                                best_sol = new_sol
                                best_cost = cost
    
    return best_sol

def apply_all_operators(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    current = copy_solution(sol)
    
    # 2-opt sur chaque route
    for i in range(len(current)):
        current[i] = two_opt(current[i], vrp.distances)
    
    # Cross-exchange
    current = cross_exchange(current, vrp)
    
    # Ejection chain
    current = ejection_chain(current, vrp)
    
    # String relocation
    current = string_relocation(current, vrp)
    
    return current
def strong_perturbation(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    """
    Opérateur fort et adapté à F-n135-k7 :
    - 40% : ejection_chain (déplacement inter-route)
    - 40% : cross_exchange (échange de segments)
    - 20% : string_relocation (déplacement de segment)
    + toujours 2-opt en post-optimisation
    """
    current = copy_solution(sol)
    
    r = random.random()
    if r < 0.4:
        current = ejection_chain(current, vrp)
    elif r < 0.8:
        current = cross_exchange(current, vrp)
    else:
        current = string_relocation(current, vrp)
    
    # Post-optimisation locale
    for i in range(len(current)):
        if len(current[i]) > 2:
            current[i] = two_opt(current[i], vrp.distances)
    
    return current
def variable_neighborhood_search(vrp: VRPInstance, initial_sol: List[List[int]], 
                                 max_time: float = 20.0) -> List[List[int]]:
    print("  → Variable Neighborhood Search (VNS)...")

    current = copy_solution(initial_sol)
    best = copy_solution(current)
    best_cost = calculate_total_cost(best, vrp.distances)

    start_time = time.time()
    neighborhood = 1  # Commence par le voisinage 1
    max_neighborhood = 4  # 1: 2-opt, 2: ejection, 3: string, 4: cross

    while (time.time() - start_time) < max_time:
        # Générer un voisin selon le voisinage courant
        if neighborhood == 1:
            candidate = copy_solution(current)
            for i in range(len(candidate)):
                candidate[i] = two_opt(candidate[i], vrp.distances)
        elif neighborhood == 2:
            candidate = ejection_chain(current, vrp)
        elif neighborhood == 3:
            candidate = string_relocation(current, vrp)
        elif neighborhood == 4:
            candidate = cross_exchange(current, vrp)
        else:
            candidate = current

        candidate_cost = calculate_total_cost(candidate, vrp.distances)

        if candidate_cost < best_cost:
            best = copy_solution(candidate)
            best_cost = candidate_cost
            current = candidate
            neighborhood = 1  # Revenir au voisinage 1 (restart)
            print(f"    VNS amélioration: {best_cost:.2f}")
        else:
            neighborhood += 1
            if neighborhood > max_neighborhood:
                neighborhood = 1
                # Perturber la solution pour échapper au local
                current = copy_solution(best)
                # Simple perturbation : shuffle une route
                if len(current) > 1:
                    i = random.randint(0, len(current) - 1)
                    if len(current[i]) > 2:
                        random.shuffle(current[i])

    return best
# ============================================================================
# 5. RECUIT SIMULÉ ULTRA-OPTIMISÉ
# ============================================================================

def simulated_annealing_ultra(vrp: VRPInstance, initial_sol: List[List[int]], 
                               max_time: int = 30) -> List[List[int]]:
    print("  → Recuit Simulé Ultra-Optimisé...")
    
    current = copy_solution(initial_sol)
    best = copy_solution(current)
    current_cost = calculate_total_cost(current, vrp.distances)
    best_cost = current_cost
    
    T = 1000.0
    T_min = 0.1
    alpha = 0.998
    
    start_time = time.time()
    
    while T > T_min and (time.time() - start_time) < max_time :
        # Appliquer tous les opérateurs
        neighbor = apply_all_operators(current, vrp)
        neighbor_cost = calculate_total_cost(neighbor, vrp.distances)
        
        delta = neighbor_cost - current_cost
        
        if delta < 0 or random.random() < math.exp(-delta / T):
            current = neighbor
            current_cost = neighbor_cost
            
            if current_cost < best_cost:
                best = copy_solution(current)
                best_cost = current_cost
                print(f"    Nouvelle meilleure solution: {best_cost:.2f}")
        
        T *= alpha
        
        elapsed = time.time() - start_time
        if (time.time() - start_time) % 10 < 0.5:
            print(f"    Temps écoulé: {elapsed:.1f}s, T={T:.2f}, Meilleur={best_cost:.2f}")
    
    return best
    

# ============================================================================
# 6. TABU SEARCH OPTIMISÉ
# ============================================================================

def tabu_search_optimized(vrp: VRPInstance, initial_sol: List[List[int]], 
                          max_iter: int = 500) -> List[List[int]]:
    print("  → Tabu Search Optimisé (version légère)...")
    
    current = copy_solution(initial_sol)
    best = copy_solution(current)
    current_cost = calculate_total_cost(current, vrp.distances)
    best_cost = current_cost
    
    tabu_list = []
    tabu_tenure = 15 # réduit
    no_improvement = 0
    for iteration in range(max_iter):
        neighbors = []
        # Générer 5 voisins légers (au lieu de 15 lourds)
        for _ in range(12):
            neighbor = generate_diverse_neighbor(current, vrp)
            cost = calculate_total_cost(neighbor, vrp.distances)
            neighbors.append((cost, neighbor))
        
        neighbors.sort()
        
        # Choisir le meilleur non-tabou
        accepted = False
        for cost, neighbor in neighbors:
            # Utilise une clé plus stable que hash(str(...))
            key = tuple(tuple(r) for r in neighbor)
            is_tabu = key in tabu_list
            if not is_tabu  or cost < best_cost:
                current = neighbor
                current_cost = cost
                tabu_list.append(key)
                if len(tabu_list) > tabu_tenure:
                    tabu_list.pop(0)
                
                if current_cost < best_cost:
                    best = copy_solution(current)
                    best_cost = current_cost
                    no_improvement = 0
                    print(f"    Itération {iteration}: Nouvelle meilleure solution: {best_cost:.2f}")
                else:
                    no_improvement += 1
                accepted = True
                break
        
        if not accepted and neighbors:
            # Accepter le meilleur même s'il est tabou (désespoir)
            cost, neighbor = neighbors[0]
            current = neighbor
            current_cost = cost
            no_improvement += 1
        if no_improvement > 40:
           print(f"    ⚠️ Diversification à l'itération {iteration}")
           current = diversification_restart(best, vrp)
           current_cost = calculate_total_cost(current, vrp.distances)
           no_improvement = 0
           tabu_list.clear()     
        if iteration % 100 == 0:
           print(f"    Itération {iteration}, Meilleur={best_cost:.2f}")
    
    return best
def generate_diverse_neighbor(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    """Génère un voisin avec différents opérateurs aléatoires"""
    current = copy_solution(sol)
    
    r = random.random()
    
    if r < 0.25:
        # Or-opt: déplacer un segment de 1-3 clients
        current = or_opt_move(current, vrp)
    elif r < 0.50:
        # Swap inter-routes
        current = swap_between_routes(current, vrp)
    elif r < 0.75:
        # Relocate: déplacer un client vers une autre route
        current = relocate_customer(current, vrp)
    else:
        # 2-opt* inter-routes
        current = two_opt_star(current, vrp)
    
    # Toujours appliquer 2-opt intra-route
    for i in range(len(current)):
        if len(current[i]) > 2:
            current[i] = two_opt(current[i], vrp.distances)
    
    return current


def or_opt_move(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    """Or-opt: déplace un segment de 1-3 clients"""
    current = copy_solution(sol)
    
    if len(current) < 2:
        return current
    
    # Choisir une route source avec au moins 2 clients
    valid_routes = [i for i in range(len(current)) if len(current[i]) >= 2]
    if not valid_routes:
        return current
    
    src_idx = random.choice(valid_routes)
    src_route = current[src_idx]
    
    # Segment de 1-3 clients
    seg_len = min(random.randint(1, 3), len(src_route))
    start_pos = random.randint(0, len(src_route) - seg_len)
    segment = src_route[start_pos:start_pos + seg_len]
    
    # Route destination
    dest_idx = random.randint(0, len(current) - 1)
    
    # Vérifier faisabilité
    seg_demand = sum(vrp.demands[c] for c in segment)
    dest_load = sum(vrp.demands[c] for c in current[dest_idx])
    
    if src_idx == dest_idx:
        # Déplacement intra-route
        new_route = src_route[:start_pos] + src_route[start_pos + seg_len:]
        insert_pos = random.randint(0, len(new_route))
        new_route = new_route[:insert_pos] + segment + new_route[insert_pos:]
        current[src_idx] = new_route
    else:
        # Déplacement inter-route
        if dest_load + seg_demand <= vrp.capacity:
            current[src_idx] = src_route[:start_pos] + src_route[start_pos + seg_len:]
            insert_pos = random.randint(0, len(current[dest_idx]))
            current[dest_idx] = current[dest_idx][:insert_pos] + segment + current[dest_idx][insert_pos:]
    
    # Nettoyer routes vides
    current = [r for r in current if r]
    
    return current


def swap_between_routes(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    """Échange un client entre deux routes"""
    current = copy_solution(sol)
    
    if len(current) < 2:
        return current
    
    r1, r2 = random.sample(range(len(current)), 2)
    
    if not current[r1] or not current[r2]:
        return current
    
    pos1 = random.randint(0, len(current[r1]) - 1)
    pos2 = random.randint(0, len(current[r2]) - 1)
    
    c1, c2 = current[r1][pos1], current[r2][pos2]
    
    # Vérifier faisabilité
    new_load1 = sum(vrp.demands[c] for c in current[r1]) - vrp.demands[c1] + vrp.demands[c2]
    new_load2 = sum(vrp.demands[c] for c in current[r2]) - vrp.demands[c2] + vrp.demands[c1]
    
    if new_load1 <= vrp.capacity and new_load2 <= vrp.capacity:
        current[r1][pos1] = c2
        current[r2][pos2] = c1
    
    return current


def relocate_customer(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    """Déplace un client vers une autre route"""
    current = copy_solution(sol)
    
    if len(current) < 2:
        return current
    
    # Route source
    valid_src = [i for i in range(len(current)) if len(current[i]) >= 1]
    if not valid_src:
        return current
    
    src_idx = random.choice(valid_src)
    pos = random.randint(0, len(current[src_idx]) - 1)
    customer = current[src_idx][pos]
    
    # Route destination
    dest_idx = random.randint(0, len(current) - 1)
    if dest_idx == src_idx:
        dest_idx = (dest_idx + 1) % len(current)
    
    dest_load = sum(vrp.demands[c] for c in current[dest_idx])
    
    if dest_load + vrp.demands[customer] <= vrp.capacity:
        current[src_idx].pop(pos)
        insert_pos = random.randint(0, len(current[dest_idx]))
        current[dest_idx].insert(insert_pos, customer)
    
    current = [r for r in current if r]
    return current


def two_opt_star(sol: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    """2-opt* entre deux routes"""
    current = copy_solution(sol)
    
    if len(current) < 2:
        return current
    
    r1, r2 = random.sample(range(len(current)), 2)
    
    if len(current[r1]) < 2 or len(current[r2]) < 2:
        return current
    
    i = random.randint(0, len(current[r1]) - 1)
    j = random.randint(0, len(current[r2]) - 1)
    
    new_r1 = current[r1][:i+1] + current[r2][j+1:]
    new_r2 = current[r2][:j+1] + current[r1][i+1:]
    
    if (is_feasible(new_r1, vrp.demands, vrp.capacity) and 
        is_feasible(new_r2, vrp.demands, vrp.capacity)):
        current[r1] = new_r1
        current[r2] = new_r2
    
    return current


def diversification_restart(best: List[List[int]], vrp: VRPInstance) -> List[List[int]]:
    """Diversification forte: reconstruction partielle"""
    current = copy_solution(best)
    
    # Extraire 20-30% des clients aléatoirement
    all_customers = [c for route in current for c in route]
    n_remove = max(5, len(all_customers) // 4)
    
    to_remove = set(random.sample(all_customers, min(n_remove, len(all_customers))))
    
    # Retirer les clients
    for i in range(len(current)):
        current[i] = [c for c in current[i] if c not in to_remove]
    
    current = [r for r in current if r]
    
    # Réinsérer avec insertion la moins coûteuse
    for customer in to_remove:
        best_cost = float('inf')
        best_route = 0
        best_pos = 0
        
        for r_idx in range(len(current)):
            route_load = sum(vrp.demands[c] for c in current[r_idx])
            if route_load + vrp.demands[customer] > vrp.capacity:
                continue
            
            for pos in range(len(current[r_idx]) + 1):
                # Calculer coût d'insertion
                if pos == 0:
                    prev = 0
                else:
                    prev = current[r_idx][pos-1]
                
                if pos == len(current[r_idx]):
                    next_c = 0
                else:
                    next_c = current[r_idx][pos]
                
                cost = (vrp.distances[prev][customer] + 
                        vrp.distances[customer][next_c] - 
                        vrp.distances[prev][next_c])
                
                if cost < best_cost:
                    best_cost = cost
                    best_route = r_idx
                    best_pos = pos
        
        if best_cost < float('inf'):
            current[best_route].insert(best_pos, customer)
        else:
            # Nouvelle route
            current.append([customer])
    
    return current

# ============================================================================
# 7. ALGORITHME GÉNÉTIQUE OPTIMISÉ
# ============================================================================

from typing import List, Tuple

# ============================================================================
# ALGORITHME MEMÉTIQUE : AG + OPTIMISATION LOCALE INTENSIVE
# ============================================================================

def memetic_algorithm(vrp, initial_sol: List[List[int]],
                      pop_size: int = 20, generations: int = 100,
                      max_time: float = 120.0) -> List[List[int]]:
    """
    Algorithme Mémétique : CHAQUE individu est optimisé localement
    C'est la SEULE façon d'améliorer une solution déjà excellente
    """
    print(" → Algorithme Mémétique (AG + Optimisation locale INTENSIVE)...")
    import time
    start_time = time.time()
    
    # ========================================================================
    # POPULATION INITIALE avec OPTIMISATION LOCALE
    # ========================================================================
    print(" Création population avec optimisation locale...")
    population = []
    
    # 1. Solution initiale optimisée
    sol = deep_local_search(initial_sol, vrp, max_time=5.0)
    population.append(sol)
    print(f"  Initial optimisé: {calculate_total_cost(sol, vrp.distances):.2f}")
    
    # 2. Perturbations + optimisation locale (diversité)
    for i in range(pop_size - 1):
        # Perturber la meilleure solution
        sol = copy_solution(population[0])
        
        # Perturbation de force variable
        strength = (i + 1) / pop_size  # 0.05 à 1.0
        n_moves = int(5 + 20 * strength)  # 5 à 25 mouvements
        
        for _ in range(n_moves):
            sol = advanced_move(sol, vrp)
        
        # OPTIMISER localement (CRUCIAL)
        sol = deep_local_search(sol, vrp, max_time=2.0)
        population.append(sol)
        
        if (i + 1) % 5 == 0:
            cost = calculate_total_cost(sol, vrp.distances)
            print(f"  Individu {i+1}/{pop_size-1}: {cost:.2f}")
    
    # Évaluation
    best = min(population, key=lambda s: calculate_total_cost(s, vrp.distances))
    best_cost = calculate_total_cost(best, vrp.distances)
    print(f"\n Population initiale - Meilleur: {best_cost:.2f}")
    
    # ========================================================================
    # ÉVOLUTION MÉMÉTIQUE
    # ========================================================================
    stagnation = 0
    gen = 0
    
    while gen < generations and (time.time() - start_time) < max_time:
        # Évaluation
        fitness = [(calculate_total_cost(ind, vrp.distances), ind) for ind in population]
        fitness.sort()
        
        if fitness[0][0] < best_cost:
            best = copy_solution(fitness[0][1])
            best_cost = fitness[0][0]
            stagnation = 0
            print(f" 🎯 Gen {gen}: AMÉLIORATION = {best_cost:.2f}")
        else:
            stagnation += 1
        
        # Restart intelligent si stagnation
        if stagnation >= 15:
            print(f" 🔄 Diversification gen {gen}")
            # Garder top 3
            population = [ind for _, ind in fitness[:3]]
            
            # Régénérer avec perturbations fortes
            while len(population) < pop_size:
                sol = copy_solution(best)
                # Perturbation TRÈS forte
                for _ in range(random.randint(10, 20)):
                    sol = advanced_move(sol, vrp)
                # Réoptimiser
                sol = deep_local_search(sol, vrp, max_time=1.5)
                population.append(sol)
            
            stagnation = 0
            gen += 1
            continue
        
        # Nouvelle génération
        elite_size = 2
        new_population = [copy_solution(ind) for _, ind in fitness[:elite_size]]
        
        # Reproduction
        while len(new_population) < pop_size:
            # Sélection (top 50%)
            p1 = fitness[random.randint(0, pop_size//2 - 1)][1]
            p2 = fitness[random.randint(0, pop_size//2 - 1)][1]
            
            # Croisement intelligent
            if random.random() < 0.7:
                child = educated_crossover(p1, p2, vrp)
            else:
                child = copy_solution(p1)
            
            # Mutation adaptative
            if random.random() < 0.5:
                for _ in range(random.randint(2, 5)):
                    child = advanced_move(child, vrp)
            
            # LE PLUS IMPORTANT : Optimisation locale sur CHAQUE enfant
            child = deep_local_search(child, vrp, max_time=1.0)
            
            new_population.append(child)
        
        population = new_population
        gen += 1
        
        if gen % 10 == 0:
            elapsed = time.time() - start_time
            diversity = calculate_diversity_fast(population[:10])
            print(f" Gen {gen}: Best={best_cost:.2f}, Time={elapsed:.1f}s, Div={diversity:.0f}")
    
    elapsed = time.time() - start_time
    print(f"\n✅ {gen} générations en {elapsed:.1f}s - Coût final: {best_cost:.2f}")
    return best


# ============================================================================
# OPTIMISATION LOCALE PROFONDE (CŒUR DE L'ALGORITHME)
# ============================================================================

def deep_local_search(sol: List[List[int]], vrp, max_time: float = 5.0) -> List[List[int]]:
    """
    Recherche locale INTENSIVE avec plusieurs opérateurs
    C'est ici que se fait la vraie amélioration !
    """
    import time
    start = time.time()
    
    current = copy_solution(sol)
    current_cost = calculate_total_cost(current, vrp.distances)
    
    improved = True
    iteration = 0
    
    while improved and (time.time() - start) < max_time:
        improved = False
        iteration += 1
        
        # 1. 2-opt sur chaque route (rapide)
        for i in range(len(current)):
            if len(current[i]) > 3:
                new_route = two_opt_full(current[i], vrp.distances)
                if calc_route_cost(new_route, vrp.distances) < calc_route_cost(current[i], vrp.distances):
                    current[i] = new_route
                    improved = True
        
        # 2. Relocate (déplacer clients entre routes)
        for r1 in range(len(current)):
            for i in range(len(current[r1])):
                customer = current[r1][i]
                
                # Trouver MEILLEURE position dans toutes les routes
                best_cost = current_cost
                best_move = None
                
                for r2 in range(len(current)):
                    # Calculer charge
                    load_r2 = sum(vrp.demands[c] for c in current[r2])
                    
                    # Si c'est la même route
                    if r1 == r2:
                        for j in range(len(current[r2])):
                            if j != i:
                                test_sol = copy_solution(current)
                                test_sol[r1].pop(i)
                                test_sol[r1].insert(j if j < i else j - 1, customer)
                                test_cost = calculate_total_cost(test_sol, vrp.distances)
                                if test_cost < best_cost:
                                    best_cost = test_cost
                                    best_move = (r1, i, r1, j if j < i else j - 1)
                    else:
                        # Vérifier capacité
                        if load_r2 + vrp.demands[customer] <= vrp.capacity:
                            for j in range(len(current[r2]) + 1):
                                test_sol = copy_solution(current)
                                test_sol[r1].pop(i)
                                test_sol[r2].insert(j, customer)
                                test_cost = calculate_total_cost(test_sol, vrp.distances)
                                if test_cost < best_cost:
                                    best_cost = test_cost
                                    best_move = (r1, i, r2, j)
                
                if best_move:
                    r1_from, i_from, r2_to, j_to = best_move
                    c = current[r1_from].pop(i_from)
                    current[r2_to].insert(j_to, c)
                    current_cost = best_cost
                    improved = True
                    break
            
            if improved:
                break
        
        # 3. Swap (échanger 2 clients entre routes)
        if not improved:
            for r1 in range(len(current)):
                for r2 in range(r1 + 1, len(current)):
                    for i in range(len(current[r1])):
                        for j in range(len(current[r2])):
                            c1 = current[r1][i]
                            c2 = current[r2][j]
                            
                            # Vérifier faisabilité
                            load1 = sum(vrp.demands[c] for c in current[r1]) - vrp.demands[c1] + vrp.demands[c2]
                            load2 = sum(vrp.demands[c] for c in current[r2]) - vrp.demands[c2] + vrp.demands[c1]
                            
                            if load1 <= vrp.capacity and load2 <= vrp.capacity:
                                test_sol = copy_solution(current)
                                test_sol[r1][i] = c2
                                test_sol[r2][j] = c1
                                test_cost = calculate_total_cost(test_sol, vrp.distances)
                                
                                if test_cost < current_cost:
                                    current[r1][i] = c2
                                    current[r2][j] = c1
                                    current_cost = test_cost
                                    improved = True
                                    break
                        if improved:
                            break
                    if improved:
                        break
                if improved:
                    break
        
        # 4. 2-opt* entre routes (occasionnel)
        if not improved and iteration % 3 == 0:
            for r1 in range(len(current)):
                for r2 in range(r1 + 1, len(current)):
                    if len(current[r1]) >= 2 and len(current[r2]) >= 2:
                        for i in range(len(current[r1])):
                            for j in range(len(current[r2])):
                                # Échanger fins de routes
                                new_r1 = current[r1][:i+1] + current[r2][j+1:]
                                new_r2 = current[r2][:j+1] + current[r1][i+1:]
                                
                                # Vérifier faisabilité
                                load1 = sum(vrp.demands[c] for c in new_r1)
                                load2 = sum(vrp.demands[c] for c in new_r2)
                                
                                if load1 <= vrp.capacity and load2 <= vrp.capacity:
                                    test_sol = copy_solution(current)
                                    test_sol[r1] = new_r1
                                    test_sol[r2] = new_r2
                                    test_cost = calculate_total_cost(test_sol, vrp.distances)
                                    
                                    if test_cost < current_cost:
                                        current = test_sol
                                        current_cost = test_cost
                                        improved = True
                                        break
                            if improved:
                                break
                        if improved:
                            break
                if improved:
                    break
    
    # Nettoyer routes vides
    current = [r for r in current if r]
    return current


# ============================================================================
# CROISEMENT INTELLIGENT
# ============================================================================

def educated_crossover(p1: List[List[int]], p2: List[List[int]], vrp) -> List[List[int]]:
    """
    Croisement qui préserve les BONNES structures des parents
    """
    # Évaluer qualité de chaque route
    routes_with_quality = []
    
    for route in p1:
        if route:
            cost = calc_route_cost(route, vrp.distances)
            efficiency = cost / len(route)  # Coût par client
            routes_with_quality.append((efficiency, route, set(route)))
    
    for route in p2:
        if route:
            cost = calc_route_cost(route, vrp.distances)
            efficiency = cost / len(route)
            routes_with_quality.append((efficiency, route, set(route)))
    
    # Trier par qualité (meilleur = plus faible coût/client)
    routes_with_quality.sort()
    
    # Prendre les meilleures routes sans conflit
    child = []
    visited = set()
    
    for efficiency, route, route_set in routes_with_quality:
        if not route_set & visited:
            child.append(route[:])
            visited.update(route_set)
    
    # Ajouter clients manquants
    missing = set(range(1, vrp.n_customers + 1)) - visited
    for c in missing:
        # Insertion gloutonne
        best_cost = float('inf')
        best_pos = (0, 0)
        
        for r in range(len(child)):
            load = sum(vrp.demands[x] for x in child[r])
            if load + vrp.demands[c] <= vrp.capacity:
                for i in range(len(child[r]) + 1):
                    prev = child[r][i-1] if i > 0 else 0
                    nxt = child[r][i] if i < len(child[r]) else 0
                    cost = vrp.distances[prev][c] + vrp.distances[c][nxt] - vrp.distances[prev][nxt]
                    if cost < best_cost:
                        best_cost = cost
                        best_pos = (r, i)
        
        if best_cost < float('inf'):
            child[best_pos[0]].insert(best_pos[1], c)
        else:
            child.append([c])
    
    return child


# ============================================================================
# MOUVEMENT AVANCÉ
# ============================================================================

def advanced_move(sol: List[List[int]], vrp) -> List[List[int]]:
    """
    Mouvement avancé : Or-opt, swap, relocate
    """
    if not sol:
        return sol
    
    current = copy_solution(sol)
    move_type = random.randint(0, 2)
    
    if move_type == 0 and len(current) >= 2:  # Or-opt
        r1 = random.randint(0, len(current) - 1)
        if len(current[r1]) >= 2:
            size = min(random.randint(1, 3), len(current[r1]))
            pos = random.randint(0, len(current[r1]) - size)
            segment = current[r1][pos:pos+size]
            
            r2 = random.randint(0, len(current) - 1)
            seg_demand = sum(vrp.demands[c] for c in segment)
            load2 = sum(vrp.demands[c] for c in current[r2])
            
            if r1 == r2 or load2 + seg_demand <= vrp.capacity:
                current[r1] = current[r1][:pos] + current[r1][pos+size:]
                if r1 != r2:
                    insert_pos = random.randint(0, len(current[r2]))
                    current[r2] = current[r2][:insert_pos] + segment + current[r2][insert_pos:]
                else:
                    insert_pos = random.randint(0, len(current[r1]))
                    current[r1] = current[r1][:insert_pos] + segment + current[r1][insert_pos:]
    
    elif move_type == 1 and len(current) >= 2:  # Swap
        r1, r2 = random.sample(range(len(current)), 2)
        if current[r1] and current[r2]:
            i, j = random.randint(0, len(current[r1])-1), random.randint(0, len(current[r2])-1)
            current[r1][i], current[r2][j] = current[r2][j], current[r1][i]
    
    else:  # Reverse
        r = random.randint(0, len(current) - 1)
        if len(current[r]) > 2:
            i = random.randint(0, len(current[r]) - 2)
            j = random.randint(i + 1, len(current[r]))
            current[r][i:j] = current[r][i:j][::-1]
    
    return [r for r in current if r]


# ============================================================================
# UTILITAIRES
# ============================================================================

def two_opt_full(route: List[int], distances: np.ndarray) -> List[int]:
    """2-opt complet"""
    best = route[:]
    improved = True
    
    while improved:
        improved = False
        best_cost = calc_route_cost(best, distances)
        
        for i in range(len(best) - 1):
            for j in range(i + 2, len(best)):
                new = best[:i+1] + best[i+1:j+1][::-1] + best[j+1:]
                new_cost = calc_route_cost(new, distances)
                if new_cost < best_cost:
                    best = new
                    best_cost = new_cost
                    improved = True
                    break
            if improved:
                break
    
    return best


def calc_route_cost(route, distances):
    if not route:
        return 0
    cost = distances[0][route[0]]
    for i in range(len(route) - 1):
        cost += distances[route[i]][route[i+1]]
    cost += distances[route[-1]][0]
    return cost


def calculate_total_cost(sol, distances):
    return sum(calc_route_cost(r, distances) for r in sol)


def copy_solution(sol):
    return [r[:] for r in sol]


def calculate_diversity_fast(population):
    if len(population) < 2:
        return 0
    diffs = 0
    for i in range(min(5, len(population))):
        for j in range(i+1, min(5, len(population))):
            flat1 = [c for r in population[i] for c in r]
            flat2 = [c for r in population[j] for c in r]
            diffs += sum(1 for a,b in zip(flat1, flat2) if a != b)
    return diffs / 10 if len(population) >= 2 else 0





# ============================================================================
# OPÉRATEURS ULTRA-RAPIDES
# ============================================================================

def fast_crossover(p1: List[List[int]], p2: List[List[int]], vrp) -> List[List[int]]:
    """
    Croisement ultra-rapide : prendre routes alternées
    """
    child = []
    visited = set()
    
    # Alterner routes des 2 parents
    all_routes = []
    for r in p1:
        all_routes.append((r, set(r)))
    for r in p2:
        all_routes.append((r, set(r)))
    
    random.shuffle(all_routes)
    
    for route, route_set in all_routes:
        if not route_set & visited:  # Pas de conflit
            child.append(route[:])
            visited.update(route_set)
    
    # Clients manquants : insertion rapide
    missing = set(range(1, vrp.n_customers + 1)) - visited
    for c in missing:
        if child:
            r = random.randint(0, len(child) - 1)
            load = sum(vrp.demands[x] for x in child[r])
            if load + vrp.demands[c] <= vrp.capacity:
                child[r].append(c)
            else:
                child.append([c])
        else:
            child.append([c])
    
    return child


def fast_mutation(sol: List[List[int]], vrp) -> List[List[int]]:
    """
    Mutation ultra-rapide : UN SEUL opérateur simple
    """
    if not sol:
        return sol
    
    current = copy_solution(sol)
    op = random.randint(0, 3)
    
    if op == 0:  # Swap intra-route
        if current:
            r = random.randint(0, len(current) - 1)
            if len(current[r]) >= 2:
                i, j = random.sample(range(len(current[r])), 2)
                current[r][i], current[r][j] = current[r][j], current[r][i]
    
    elif op == 1:  # Swap inter-routes
        if len(current) >= 2:
            r1, r2 = random.sample(range(len(current)), 2)
            if current[r1] and current[r2]:
                i = random.randint(0, len(current[r1]) - 1)
                j = random.randint(0, len(current[r2]) - 1)
                current[r1][i], current[r2][j] = current[r2][j], current[r1][i]
    
    elif op == 2:  # Déplacer un client
        if len(current) >= 2:
            r1 = random.randint(0, len(current) - 1)
            if current[r1]:
                i = random.randint(0, len(current[r1]) - 1)
                c = current[r1].pop(i)
                r2 = random.randint(0, len(current) - 1)
                j = random.randint(0, len(current[r2]))
                current[r2].insert(j, c)
    
    else:  # Inverser segment
        if current:
            r = random.randint(0, len(current) - 1)
            if len(current[r]) > 2:
                i = random.randint(0, len(current[r]) - 2)
                j = random.randint(i + 1, len(current[r]))
                current[r][i:j] = current[r][i:j][::-1]
    
    return [r for r in current if r]


def fast_perturbation(sol: List[List[int]], vrp) -> List[List[int]]:
    """
    Perturbation rapide : swap + reverse
    """
    current = copy_solution(sol)
    
    # 2-3 opérations aléatoires
    for _ in range(random.randint(2, 3)):
        if len(current) >= 2 and current[0]:
            r1, r2 = random.sample(range(len(current)), 2)
            if current[r1] and current[r2]:
                i = random.randint(0, len(current[r1]) - 1)
                j = random.randint(0, len(current[r2]) - 1)
                current[r1][i], current[r2][j] = current[r2][j], current[r1][i]
    
    return [r for r in current if r]


def two_opt_fast(route: List[int], distances: np.ndarray) -> List[int]:
    """
    2-opt rapide : seulement 1 passe, pas de boucle while
    """
    best = route[:]
    best_cost = calc_route_cost(best, distances)
    
    for i in range(len(route) - 1):
        for j in range(i + 2, min(i + 20, len(route))):  # Limité à 20 voisins
            new = route[:i+1] + route[i+1:j+1][::-1] + route[j+1:]
            new_cost = calc_route_cost(new, distances)
            if new_cost < best_cost:
                return new  # Retour immédiat
    
    return best


def calc_route_cost(route, distances):
    """Calcul rapide du coût d'une route"""
    if not route:
        return 0
    cost = distances[0][route[0]]
    for i in range(len(route) - 1):
        cost += distances[route[i]][route[i+1]]
    cost += distances[route[-1]][0]
    return cost


# ============================================================================
# CONSTRUCTIONS RAPIDES
# ============================================================================

def nearest_neighbor_random_start(vrp) -> List[List[int]]:
    """NN rapide depuis un client aléatoire"""
    unvisited = set(range(1, vrp.n_customers + 1))
    solution = []
    
    while unvisited:
        route = []
        load = 0
        
        # Départ aléatoire
        current = random.choice(list(unvisited))
        route.append(current)
        load = vrp.demands[current]
        unvisited.remove(current)
        
        # NN classique
        while unvisited:
            best = None
            best_dist = float('inf')
            
            for node in unvisited:
                if load + vrp.demands[node] <= vrp.capacity:
                    dist = vrp.distances[current][node]
                    if dist < best_dist:
                        best_dist = dist
                        best = node
            
            if best is None:
                break
            
            route.append(best)
            load += vrp.demands[best]
            current = best
            unvisited.remove(best)
        
        solution.append(route)
    
    return solution


def random_greedy_solution(vrp) -> List[List[int]]:
    """
    Solution gloutonne avec choix aléatoire parmi top K
    """
    unvisited = set(range(1, vrp.n_customers + 1))
    solution = []
    
    while unvisited:
        route = []
        load = 0
        current = 0
        
        while unvisited:
            # Trouver top 5 plus proches
            candidates = []
            for node in unvisited:
                if load + vrp.demands[node] <= vrp.capacity:
                    dist = vrp.distances[current][node]
                    candidates.append((dist, node))
            
            if not candidates:
                break
            
            candidates.sort()
            # Choisir aléatoirement dans top 5
            best = random.choice(candidates[:min(5, len(candidates))])[1]
            
            route.append(best)
            load += vrp.demands[best]
            current = best
            unvisited.remove(best)
        
        if route:
            solution.append(route)
    
    return solution


# ============================================================================
# UTILITAIRES
# ============================================================================

def copy_solution(sol):
    return [r[:] for r in sol]


def calculate_total_cost(sol, distances):
    return sum(calc_route_cost(r, distances) for r in sol)


# ============================================================================
# VERSION HYBRIDE : AG + RECHERCHE LOCALE
# ============================================================================

def genetic_with_local_search(vrp, initial_sol: List[List[int]],
                              pop_size: int = 25, generations: int = 300,
                              max_time: float = 120.0) -> List[List[int]]:
    """
    AG rapide avec recherche locale intensive sur le meilleur
    """
    print(" → AG Hybride (AG rapide + Recherche locale)...")
    import time
    start_time = time.time()
    
    # AG rapide (80% du temps)
    time_ag = max_time * 0.8
    best_ag = genetic_algorithm_fast(vrp, initial_sol, 
                                     pop_size=pop_size, 
                                     generations=generations,
                                     max_time=time_ag)
    
    # Recherche locale intensive (20% du temps)
    print("\n → Recherche locale intensive...")
    best = copy_solution(best_ag)
    best_cost = calculate_total_cost(best, vrp.distances)
    
    time_left = max_time - (time.time() - start_time)
    local_start = time.time()
    
    iteration = 0
    while (time.time() - local_start) < time_left:
        # 2-opt sur toutes les routes
        improved = False
        for i in range(len(best)):
            if len(best[i]) > 3:
                new_route = two_opt_full(best[i], vrp.distances)
                if calc_route_cost(new_route, vrp.distances) < calc_route_cost(best[i], vrp.distances):
                    best[i] = new_route
                    improved = True
        
        # Relocate inter-routes
        for _ in range(5):  # 5 tentatives
            if len(best) >= 2:
                r1 = random.randint(0, len(best) - 1)
                if best[r1]:
                    i = random.randint(0, len(best[r1]) - 1)
                    c = best[r1].pop(i)
                    
                    best_pos = (0, 0)
                    best_delta = float('inf')
                    
                    for r2 in range(len(best)):
                        load = sum(vrp.demands[x] for x in best[r2])
                        if load + vrp.demands[c] <= vrp.capacity:
                            for j in range(len(best[r2]) + 1):
                                prev = best[r2][j-1] if j > 0 else 0
                                nxt = best[r2][j] if j < len(best[r2]) else 0
                                delta = vrp.distances[prev][c] + vrp.distances[c][nxt] - vrp.distances[prev][nxt]
                                if delta < best_delta:
                                    best_delta = delta
                                    best_pos = (r2, j)
                    
                    best[best_pos[0]].insert(best_pos[1], c)
        
        new_cost = calculate_total_cost(best, vrp.distances)
        if new_cost < best_cost:
            best_cost = new_cost
            print(f"  Itération {iteration}: {best_cost:.2f}")
            improved = True
        
        if not improved:
            break
        
        iteration += 1
    
    print(f"\n✅ Coût final: {best_cost:.2f}")
    return best


def two_opt_full(route: List[int], distances: np.ndarray) -> List[int]:
    """2-opt complet avec boucle"""
    best = route[:]
    improved = True
    
    while improved:
        improved = False
        best_cost = calc_route_cost(best, distances)
        
        for i in range(len(best) - 1):
            for j in range(i + 2, len(best)):
                new = best[:i+1] + best[i+1:j+1][::-1] + best[j+1:]
                new_cost = calc_route_cost(new, distances)
                if new_cost < best_cost:
                    best = new
                    best_cost = new_cost
                    improved = True
                    break
            if improved:
                break
    
    return best




# ============================================================================
# OPÉRATEURS GÉNÉTIQUES AMÉLIORÉS
# ============================================================================

def order_crossover(parent1: List[List[int]], parent2: List[List[int]], 
                   vrp) -> List[List[int]]:
    """
    Order Crossover (OX) adapté au VRP multi-routes
    """
    # Convertir en séquences plates
    seq1 = [c for route in parent1 for c in route]
    seq2 = [c for route in parent2 for c in route]
    
    if len(seq1) < 2:
        return copy_solution(parent1)
    
    # Choisir segment aléatoire
    cut1 = random.randint(0, len(seq1) - 1)
    cut2 = random.randint(cut1 + 1, len(seq1))
    
    # Créer l'enfant
    child_seq = [None] * len(seq1)
    child_seq[cut1:cut2] = seq1[cut1:cut2]
    
    # Remplir avec parent2
    pos = cut2
    for gene in seq2[cut2:] + seq2[:cut2]:
        if gene not in child_seq:
            if pos >= len(child_seq):
                pos = 0
            child_seq[pos] = gene
            pos += 1
    
    # Reconstruire les routes
    child = reconstruct_routes(child_seq, vrp)
    return child


def mega_perturbation(sol: List[List[int]], vrp) -> List[List[int]]:
    """
    Perturbation TRÈS forte - destruction/reconstruction
    """
    if not sol:
        return sol
    
    current = copy_solution(sol)
    method = random.choice(['destroy_repair', 'route_swap', 'shuffle_multi'])
    
    if method == 'destroy_repair':
        # Détruire 30-50% et reconstruire
        all_customers = [c for route in current for c in route]
        n_remove = max(5, len(all_customers) // 3)
        to_remove = set(random.sample(all_customers, min(n_remove, len(all_customers))))
        
        # Retirer
        for i in range(len(current)):
            current[i] = [c for c in current[i] if c not in to_remove]
        current = [r for r in current if r]
        
        # Réinsérer glouton
        for customer in to_remove:
            current = greedy_insert(customer, current, vrp)
    
    elif method == 'route_swap':
        # Échanger 2-3 routes complètes
        if len(current) >= 2:
            n_swap = min(3, len(current))
            routes = random.sample(range(len(current)), n_swap)
            random.shuffle(routes)
            swapped = [current[i] for i in routes]
            for i, route_idx in enumerate(routes):
                current[route_idx] = swapped[i]
    
    else:  # shuffle_multi
        # Mélanger plusieurs routes
        n_shuffle = min(random.randint(2, 4), len(current))
        for _ in range(n_shuffle):
            if current:
                idx = random.randint(0, len(current) - 1)
                if len(current[idx]) > 1:
                    random.shuffle(current[idx])
    
    return current


def light_mutation(sol: List[List[int]], vrp) -> List[List[int]]:
    """
    Mutation légère - un seul opérateur
    """
    current = copy_solution(sol)
    op = random.choice(['swap', 'insert', 'reverse', '2opt'])
    
    if op == 'swap' and len(current) >= 2:
        # Swap entre routes
        r1, r2 = random.sample(range(len(current)), 2)
        if current[r1] and current[r2]:
            i, j = random.randint(0, len(current[r1])-1), random.randint(0, len(current[r2])-1)
            current[r1][i], current[r2][j] = current[r2][j], current[r1][i]
    
    elif op == 'insert' and current:
        # Déplacer un client
        r1 = random.randint(0, len(current)-1)
        if len(current[r1]) > 1:
            i = random.randint(0, len(current[r1])-1)
            customer = current[r1].pop(i)
            r2 = random.randint(0, len(current)-1)
            j = random.randint(0, len(current[r2]))
            current[r2].insert(j, customer)
    
    elif op == 'reverse' and current:
        # Inverser segment
        r = random.randint(0, len(current)-1)
        if len(current[r]) > 2:
            i = random.randint(0, len(current[r])-2)
            j = random.randint(i+1, len(current[r]))
            current[r][i:j] = current[r][i:j][::-1]
    
    elif op == '2opt' and current:
        # 2-opt sur une route
        r = random.randint(0, len(current)-1)
        if len(current[r]) > 3:
            current[r] = two_opt(current[r], vrp.distances)
    
    current = [r for r in current if r]
    return current


# ============================================================================
# FONCTIONS DE CONSTRUCTION DIVERSIFIÉES
# ============================================================================

def nearest_neighbor_random_start(vrp) -> List[List[int]]:
    """NN depuis un client aléatoire"""
    unvisited = set(range(1, vrp.n_customers + 1))
    solution = []
    
    while unvisited:
        route = []
        load = 0
        current = random.choice(list(unvisited)) if unvisited else 0
        
        if current != 0:
            route.append(current)
            load = vrp.demands[current]
            unvisited.remove(current)
        
        while unvisited:
            best = None
            best_dist = float('inf')
            for node in unvisited:
                if load + vrp.demands[node] <= vrp.capacity:
                    dist = vrp.distances[current][node]
                    if dist < best_dist:
                        best_dist = dist
                        best = node
            
            if best is None:
                break
            
            route.append(best)
            load += vrp.demands[best]
            current = best
            unvisited.remove(best)
        
        if route:
            solution.append(route)
    
    return solution


def farthest_insertion(vrp) -> List[List[int]]:
    """Insertion du plus éloigné"""
    unvisited = set(range(1, vrp.n_customers + 1))
    solution = []
    
    while unvisited:
        route = []
        load = 0
        
        # Client le plus éloigné du dépôt
        farthest = max(unvisited, key=lambda c: vrp.distances[0][c])
        route.append(farthest)
        load = vrp.demands[farthest]
        unvisited.remove(farthest)
        
        while unvisited:
            # Trouver le plus éloigné de la route
            best = None
            best_dist = -1
            for node in unvisited:
                if load + vrp.demands[node] > vrp.capacity:
                    continue
                min_dist = min(vrp.distances[node][c] for c in route)
                if min_dist > best_dist:
                    best_dist = min_dist
                    best = node
            
            if best is None:
                break
            
            route.append(best)
            load += vrp.demands[best]
            unvisited.remove(best)
        
        if route:
            solution.append(route)
    
    return solution


def random_savings(vrp) -> List[List[int]]:
    """Clarke-Wright avec ordre aléatoire des savings"""
    savings = []
    for i in range(1, vrp.n_customers + 1):
        for j in range(i + 1, vrp.n_customers + 1):
            s = vrp.distances[0][i] + vrp.distances[0][j] - vrp.distances[i][j]
            savings.append((s, i, j))
    
    random.shuffle(savings)  # ALÉATOIRE
    
    routes = [[i] for i in range(1, vrp.n_customers + 1)]
    loads = [vrp.demands[i] for i in range(1, vrp.n_customers + 1)]
    node_map = {i: i-1 for i in range(1, vrp.n_customers + 1)}
    
    for s, i, j in savings[:len(savings)//2]:  # Utiliser 50% des savings
        ri, rj = node_map[i], node_map[j]
        if ri != rj and loads[ri] + loads[rj] <= vrp.capacity:
            routes[ri].extend(routes[rj])
            loads[ri] += loads[rj]
            routes[rj] = []
            loads[rj] = 0
            for node in routes[ri]:
                node_map[node] = ri
    
    return [r for r in routes if r]


def random_solution(vrp) -> List[List[int]]:
    """Solution complètement aléatoire (mais valide)"""
    customers = list(range(1, vrp.n_customers + 1))
    random.shuffle(customers)
    return reconstruct_routes(customers, vrp)


# ============================================================================
# UTILITAIRES
# ============================================================================

def reconstruct_routes(customers: List[int], vrp) -> List[List[int]]:
    """Reconstruit des routes valides depuis une séquence"""
    routes = []
    route = []
    load = 0
    
    for c in customers:
        if load + vrp.demands[c] <= vrp.capacity:
            route.append(c)
            load += vrp.demands[c]
        else:
            if route:
                routes.append(route)
            route = [c]
            load = vrp.demands[c]
    
    if route:
        routes.append(route)
    
    return routes


def greedy_insert(customer: int, routes: List[List[int]], vrp) -> List[List[int]]:
    """Insère un client à la meilleure position"""
    best_cost = float('inf')
    best_route = 0
    best_pos = 0
    
    for r_idx in range(len(routes)):
        load = sum(vrp.demands[c] for c in routes[r_idx])
        if load + vrp.demands[customer] > vrp.capacity:
            continue
        
        for pos in range(len(routes[r_idx]) + 1):
            prev = routes[r_idx][pos-1] if pos > 0 else 0
            nxt = routes[r_idx][pos] if pos < len(routes[r_idx]) else 0
            cost = vrp.distances[prev][customer] + vrp.distances[customer][nxt] - vrp.distances[prev][nxt]
            
            if cost < best_cost:
                best_cost = cost
                best_route = r_idx
                best_pos = pos
    
    if best_cost < float('inf'):
        routes[best_route].insert(best_pos, customer)
    else:
        routes.append([customer])
    
    return routes


def repair_solution(sol: List[List[int]], vrp) -> List[List[int]]:
    """Répare une solution invalide"""
    # Vérifier doublons
    all_customers = [c for route in sol for c in route]
    if len(all_customers) != len(set(all_customers)):
        # Reconstruire complètement
        unique = list(set(all_customers))
        missing = set(range(1, vrp.n_customers + 1)) - set(unique)
        all_customers = unique + list(missing)
        random.shuffle(all_customers)
        return reconstruct_routes(all_customers, vrp)
    
    # Vérifier capacités
    fixed = []
    for route in sol:
        load = sum(vrp.demands[c] for c in route)
        if load <= vrp.capacity:
            fixed.append(route)
        else:
            # Diviser la route
            fixed.extend(reconstruct_routes(route, vrp))
    
    return fixed


def local_search_light(sol: List[List[int]], vrp) -> List[List[int]]:
    """Recherche locale légère"""
    current = copy_solution(sol)
    for i in range(len(current)):
        if len(current[i]) > 3:
            current[i] = two_opt(current[i], vrp.distances)
    return current


def tournament_selection(fitness: List[Tuple[float, List]], k: int = 5):
    """Sélection par tournoi"""
    tournament = random.sample(fitness, min(k, len(fitness)))
    return min(tournament, key=lambda x: x[0])[1]


def calculate_diversity(population: List[List[List[int]]], vrp) -> float:
    """Mesure la diversité"""
    if len(population) < 2:
        return 0.0
    
    total = 0
    count = 0
    for i in range(min(10, len(population))):
        for j in range(i+1, min(10, len(population))):
            flat1 = [c for r in population[i] for c in r]
            flat2 = [c for r in population[j] for c in r]
            diff = sum(1 for a, b in zip(flat1, flat2) if a != b)
            total += diff
            count += 1
    
    return total / count if count > 0 else 0.0


def copy_solution(sol):
    return [r[:] for r in sol]


# Fonctions manquantes (à garder de votre code)
def calculate_total_cost(sol, distances):
    def calc_route(route):
        if not route:
            return 0
        cost = distances[0][route[0]]
        for i in range(len(route)-1):
            cost += distances[route[i]][route[i+1]]
        cost += distances[route[-1]][0]
        return cost
    return sum(calc_route(r) for r in sol)


def two_opt(route, distances):
    """2-opt simple"""
    best = route[:]
    improved = True
    while improved:
        improved = False
        for i in range(len(best) - 1):
            for j in range(i + 2, len(best)):
                new = best[:i+1] + best[i+1:j+1][::-1] + best[j+1:]
                if calculate_total_cost([new], distances) < calculate_total_cost([best], distances):
                    best = new
                    improved = True
                    break
            if improved:
                break
    return best
# ============================================================================
# PHASE B : HYBRIDATIONS DOUBLES
# ============================================================================

def genetic_then_tabu(vrp, initial_sol, max_time=60.0):
    """Génétique → Tabu"""
    print("  → Génétique → Tabu...")
    start = time.time()
    
    # Phase 1 : Génétique (60% du temps)
    sol_gen = memetic_algorithm(vrp, initial_sol, 
                                pop_size=15, 
                                generations=50,
                                max_time=max_time * 0.6)
    
    # Phase 2 : Tabu (40% du temps)
    elapsed = time.time() - start
    remaining_time = max_time - elapsed
    max_iter = int(remaining_time * 20)  # ~20 itérations/seconde
    sol_tabu = tabu_search_optimized(vrp, sol_gen, max_iter=max_iter)
    
    return sol_tabu


def tabu_then_genetic(vrp, initial_sol, max_time=60.0):
    """Tabu → Génétique (ordre inverse)"""
    print("  → Tabu → Génétique...")
    start = time.time()
    
    # Phase 1 : Tabu (40% du temps)
    max_iter = int(max_time * 0.4 * 20)
    sol_tabu = tabu_search_optimized(vrp, initial_sol, max_iter=max_iter)
    
    # Phase 2 : Génétique (60% du temps)
    elapsed = time.time() - start
    remaining_time = max_time - elapsed
    sol_gen = memetic_algorithm(vrp, sol_tabu,
                               pop_size=15,
                               generations=50,
                               max_time=remaining_time)
    
    return sol_gen


def genetic_then_vns(vrp, initial_sol, max_time=60.0):
    """Génétique → VNS"""
    print("  → Génétique → VNS...")
    start = time.time()
    
    # Phase 1 : Génétique (70% du temps)
    sol_gen = memetic_algorithm(vrp, initial_sol,
                                pop_size=15,
                                generations=50,
                                max_time=max_time * 0.7)
    
    # Phase 2 : VNS (30% du temps)
    elapsed = time.time() - start
    remaining_time = max_time - elapsed
    sol_vns = variable_neighborhood_search(vrp, sol_gen, max_time=remaining_time)
    
    return sol_vns


def vns_then_genetic(vrp, initial_sol, max_time=60.0):
    """VNS → Génétique"""
    print("  → VNS → Génétique...")
    start = time.time()
    
    # Phase 1 : VNS (30% du temps)
    sol_vns = variable_neighborhood_search(vrp, initial_sol, max_time=max_time * 0.3)
    
    # Phase 2 : Génétique (70% du temps)
    elapsed = time.time() - start
    remaining_time = max_time - elapsed
    sol_gen = memetic_algorithm(vrp, sol_vns,
                               pop_size=15,
                               generations=50,
                               max_time=remaining_time)
    
    return sol_gen


# ============================================================================
# PHASE C : HYBRIDATIONS TRIPLES
# ============================================================================

def genetic_tabu_vns(vrp, initial_sol, max_time=90.0):
    """Génétique → Tabu → VNS"""
    print("  → Génétique → Tabu → VNS...")
    start = time.time()
    
    # Phase 1 : Génétique (50% du temps)
    sol_gen = memetic_algorithm(vrp, initial_sol,
                                pop_size=15,
                                generations=50,
                                max_time=max_time * 0.5)
    
    # Phase 2 : Tabu (30% du temps)
    elapsed = time.time() - start
    max_iter = int((max_time * 0.3) * 20)
    sol_tabu = tabu_search_optimized(vrp, sol_gen, max_iter=max_iter)
    
    # Phase 3 : VNS (20% du temps)
    elapsed = time.time() - start
    remaining_time = max_time - elapsed
    sol_vns = variable_neighborhood_search(vrp, sol_tabu, max_time=remaining_time)
    
    return sol_vns


def tabu_genetic_tabu(vrp, initial_sol, max_time=90.0):
    """Tabu → Génétique → Tabu (raffinement itératif)"""
    print("  → Tabu → Génétique → Tabu...")
    start = time.time()
    
    # Phase 1 : Tabu initial (30% du temps)
    max_iter1 = int((max_time * 0.3) * 20)
    sol_tabu1 = tabu_search_optimized(vrp, initial_sol, max_iter=max_iter1)
    
    # Phase 2 : Génétique (40% du temps)
    elapsed = time.time() - start
    remaining = max_time - elapsed
    sol_gen = memetic_algorithm(vrp, sol_tabu1,
                                pop_size=15,
                                generations=50,
                                max_time=remaining * 0.6)
    
    # Phase 3 : Tabu final (30% du temps)
    elapsed = time.time() - start
    remaining_time = max_time - elapsed
    max_iter2 = int(remaining_time * 20)
    sol_tabu2 = tabu_search_optimized(vrp, sol_gen, max_iter=max_iter2)
    
    return sol_tabu2


def vns_genetic_tabu(vrp, initial_sol, max_time=90.0):
    """VNS → Génétique → Tabu"""
    print("  → VNS → Génétique → Tabu...")
    start = time.time()
    
    # Phase 1 : VNS (25% du temps)
    sol_vns = variable_neighborhood_search(vrp, initial_sol, max_time=max_time * 0.25)
    
    # Phase 2 : Génétique (45% du temps)
    elapsed = time.time() - start
    remaining = max_time - elapsed
    sol_gen = memetic_algorithm(vrp, sol_vns,
                                pop_size=15,
                                generations=50,
                                max_time=remaining * 0.6)
    
    # Phase 3 : Tabu (30% du temps)
    elapsed = time.time() - start
    remaining_time = max_time - elapsed
    max_iter = int(remaining_time * 20)
    sol_tabu = tabu_search_optimized(vrp, sol_gen, max_iter=max_iter)
    
    return sol_tabu


# ============================================================================
# PHASE D : HYBRIDATION ADAPTATIVE (CHAMPION) 🏆
# ============================================================================

def adaptive_genetic_tabu(vrp, initial_sol, max_time=120.0):
    """
    🏆 CHAMPION : Génétique ↔ Tabu Adaptatif
    Alterne entre exploration (génétique) et intensification (tabu)
    jusqu'à convergence optimale.
    """
    print("  → 🏆 HYBRIDATION ADAPTATIVE (Génétique ↔ Tabu)...")
    start_time = time.time()
    
    current = copy_solution(initial_sol)
    best = copy_solution(current)
    best_cost = calculate_total_cost(best, vrp.distances)
    
    phase = 'genetic'  # Commence par génétique
    stagnation = 0
    iteration = 0
    
    # Historique pour détecter convergence
    cost_history = [best_cost]
    
    while (time.time() - start_time) < max_time:
        iteration += 1
        elapsed = time.time() - start_time
        remaining = max_time - elapsed
        
        if remaining < 5.0:  # Moins de 5s restantes
            break
        
        print(f"\n  🔄 Itération {iteration} - Phase: {phase.upper()} - Best: {best_cost:.2f}")
        
        # ====================================================================
        # ALTERNANCE ADAPTATIVE
        # ====================================================================
        
        if phase == 'genetic':
            # EXPLORATION : Génétique pour diversifier
            time_budget = min(remaining * 0.4, 30.0)
            current = memetic_algorithm(vrp, current,
                                       pop_size=12,
                                       generations=30,
                                       max_time=time_budget)
            phase = 'tabu'  # Prochain : intensification
            
        else:  # phase == 'tabu'
            # INTENSIFICATION : Tabu pour affiner
            time_budget = min(remaining * 0.3, 20.0)
            max_iter = int(time_budget * 20)
            current = tabu_search_optimized(vrp, current, max_iter=max_iter)
            phase = 'genetic'  # Prochain : exploration
        
        # ====================================================================
        # ÉVALUATION ET DÉCISION
        # ====================================================================
        
        current_cost = calculate_total_cost(current, vrp.distances)
        cost_history.append(current_cost)
        
        if current_cost < best_cost:
            improvement = best_cost - current_cost
            best = copy_solution(current)
            best_cost = current_cost
            stagnation = 0
            print(f"    ✅ AMÉLIORATION : {best_cost:.2f} (gain: {improvement:.2f})")
        else:
            stagnation += 1
            print(f"    ⚪ Pas d'amélioration (stagnation: {stagnation})")
        
        # ====================================================================
        # GESTION DE LA STAGNATION
        # ====================================================================
        
        if stagnation >= 3:
            print(f"    🔥 DIVERSIFICATION FORTE (stagnation {stagnation})")
            # Perturbation massive + VNS rapide
            current = copy_solution(best)
            for _ in range(random.randint(15, 25)):
                current = advanced_move(current, vrp)
            
            # VNS rapide pour réparer
            vns_time = min(remaining * 0.15, 10.0)
            if vns_time > 2.0:
                current = variable_neighborhood_search(vrp, current, max_time=vns_time)
            
            stagnation = 0
            phase = 'genetic'  # Redémarrer en exploration
        
        # ====================================================================
        # CRITÈRE DE CONVERGENCE
        # ====================================================================
        
        if len(cost_history) >= 5:
            recent_improvement = cost_history[-5] - cost_history[-1]
            if recent_improvement < 0.5:  # Moins de 0.5 d'amélioration en 5 itérations
                print(f"    🎯 CONVERGENCE ATTEINTE (amélioration < 0.5 sur 5 iter)")
                
                # Dernier polish intensif avec VNS
                final_time = min(remaining, 15.0)
                if final_time > 3.0:
                    print(f"    ✨ POLISH FINAL avec VNS ({final_time:.1f}s)")
                    best = variable_neighborhood_search(vrp, best, max_time=final_time)
                    best_cost = calculate_total_cost(best, vrp.distances)
                
                break
    
    # ========================================================================
    # RÉSUMÉ FINAL
    # ========================================================================
    
    total_time = time.time() - start_time
    print(f"\n  🏆 TERMINÉ en {iteration} itérations ({total_time:.1f}s)")
    print(f"  📊 Coût initial: {cost_history[0]:.2f}")
    print(f"  📊 Coût final: {best_cost:.2f}")
    print(f"  📊 Amélioration totale: {cost_history[0] - best_cost:.2f}")
    
    return best



# ============================================================================
# 8. SYSTÈME PRINCIPAL
# ============================================================================
# ============================================================================
# 8. SYSTÃˆME PRINCIPAL
# ============================================================================

def main():
   
    print("="*80)
    print("SYSTÃˆME VRP ULTRA-OPTIMISÃ‰ - F-n135-k7")
    print("="*80)
    
    # Charger donnÃ©es
    vrp = VRPInstance()
    filename = "C:/Users/912ke/Desktop/opt/F-n135-k7.vrp"
    try:
        vrp.load_from_file(filename)
    except FileNotFoundError:
        print(f"\nâŒ ERREUR: Fichier '{filename}' non trouvÃ©!")
        print("\nðŸ“ Utilisation des donnÃ©es intÃ©grÃ©es Ã  la place...")
        vrp.load_f135_k7_embedded()
    except Exception as e:
        print(f"\nâŒ ERREUR lors de la lecture: {e}")
        print("\nðŸ“ Utilisation des donnÃ©es intÃ©grÃ©es Ã  la place...")
        vrp.load_f135_k7_embedded()
    
    # VÃ©rification que les donnÃ©es sont chargÃ©es
    if vrp.n_customers == 0 or len(vrp.coords) == 0:
        print("\nâŒ ERREUR CRITIQUE: Aucune donnÃ©e chargÃ©e!")
        return
    
    print(f"\nDataset: {vrp.n_customers} clients, {vrp.n_vehicles} vÃ©hicules, capacitÃ©={vrp.capacity}")
    
    results = {}
    
    # 1. Algorithmes de base
    print("\n" + "="*80)
    print("PHASE 1: ALGORITHMES DE BASE")
    print("="*80)
    
    start = time.time()
    sol1 = nearest_neighbor_improved(vrp)
    cost1 = calculate_total_cost(sol1, vrp.distances)
    results['Nearest Neighbor'] = cost1
    print(f"  CoÃ»t: {cost1:.2f} (temps: {time.time()-start:.2f}s)\n")
    
    start = time.time()
    sol2 = clarke_wright_improved(vrp)
    cost2 = calculate_total_cost(sol2, vrp.distances)
    results['Clarke-Wright'] = cost2
    print(f"  CoÃ»t: {cost2:.2f} (temps: {time.time()-start:.2f}s)\n")
    
    start = time.time()
    sol3 = sweep_algorithm(vrp)
    cost3 = calculate_total_cost(sol3, vrp.distances)
    results['Sweep'] = cost3
    print(f"  CoÃ»t: {cost3:.2f} (temps: {time.time()-start:.2f}s)\n")
    
    
    
    # 2. Recuit simulÃ© ultra-optimisÃ©
    print("\n" + "="*80)
    print("PHASE 2: RECUIT SIMULÃ‰ ULTRA-OPTIMISÃ‰")
    print("="*80)
    start = time.time()
    sol_sa = simulated_annealing_ultra(vrp,sol2, max_time=80)
    cost_sa = calculate_total_cost(sol_sa, vrp.distances)
    if cost_sa >= 1183.0:
        print(f"  âš ï¸ Recuit SimulÃ© â‰¥1183 ({cost_sa:.2f}). Passage Ã  VNS...")
        sol_sa_vns = variable_neighborhood_search(vrp,sol_sa, max_time=80)
        cost_sa_vns = calculate_total_cost(sol_sa_vns, vrp.distances)
        if cost_sa_vns < cost_sa:
            sol_sa, cost_sa = sol_sa_vns, cost_sa_vns
            results['Recuit SimulÃ© (fallback VNS)'] = cost_sa
        else:
            results['Recuit SimulÃ©'] = cost_sa
    else:
       results['Recuit SimulÃ©'] = cost_sa
    print(f"  CoÃ»t final: {cost_sa:.2f} (temps: {time.time()-start:.2f}s)\n")
    # Choisir la meilleure solution de base
    # Choisir la meilleure solution de base
    best_base = sol_sa

    # ======================================================================
    # PHASE A : MÃ‰TAHEURISTIQUES SEULES (avant hybridation)
    # ======================================================================
    print("\n" + "="*80)
    print("PHASE A: MÃ‰TAHEURISTIQUES SEULES")
    print("="*80)
   

    
    # â†’ Tabu Search seul
    print("\nâ†’ TABU SEARCH SEUL")
    start = time.time()
    sol_tabu = tabu_search_optimized(vrp, best_base, max_iter=800)  # â±ï¸ rÃ©duit
    sol_final = variable_neighborhood_search(vrp, sol_tabu, max_time=30.0)

    cost_final = calculate_total_cost(sol_final, vrp.distances)
    print(f"  CoÃ»t: {cost_final:.2f} (temps: {time.time()-start:.2f}s)\n")

    # â†’ Algorithme GÃ©nÃ©tique seul
    print("\nâ†’ ALGORITHME GÃ‰NÃ‰TIQUE SEUL")
    start = time.time()
    # MEILLEURE OPTION : AG rapide + recherche locale
    sol_memetic = memetic_algorithm(vrp, best_base, 
                                 pop_size=15,      # Plus petit
                                 generations=50,   # Moins nombreuses
                                 max_time=120.0)
    cost_gen_seul = calculate_total_cost(sol_memetic, vrp.distances)
    results['GÃ©nÃ©tique Seul'] = cost_gen_seul
    print(f"  CoÃ»t: {cost_gen_seul:.2f} (temps: {time.time()-start:.2f}s)\n")  
    
    # 3. Hybridations
    print("\n" + "="*80)
    print("PHASE 3: HYBRIDATIONS")
    print("="*80)
    
    # AlÃ©atoire + Recuit SimulÃ©
    print("\nâ†’ ALÃ‰ATOIRE + RECUIT SIMULÃ‰")
    random.shuffle(best_base)
    start = time.time()
    sol_hybrid1 = simulated_annealing_ultra(vrp, best_base, max_time=30)
    cost_hybrid1 = calculate_total_cost(sol_hybrid1, vrp.distances)
    results['AlÃ©atoire + SA'] = cost_hybrid1
    print(f"  CoÃ»t: {cost_hybrid1:.2f} (temps: {time.time()-start:.2f}s)\n")
    
    # Clarke-Wright + Recuit SimulÃ©
    print("\nâ†’ CLARKE-WRIGHT + RECUIT SIMULÃ‰")
    start = time.time()
    sol_hybrid2 = simulated_annealing_ultra(vrp, sol2, max_time=30)
    cost_hybrid2 = calculate_total_cost(sol_hybrid2, vrp.distances)
    results['CW + SA'] = cost_hybrid2
    print(f"  CoÃ»t: {cost_hybrid2:.2f} (temps: {time.time()-start:.2f}s)\n")
    
    # Tabu Search + GÃ©nÃ©tique
    print("\nâ†’ TABU SEARCH + GÃ‰NÃ‰TIQUE")
    start = time.time()
    sol_tabu = tabu_search_optimized(vrp, sol_sa, max_iter=800)
    cost_tabu = calculate_total_cost(sol_tabu, vrp.distances)
    results['Tabu Search'] = cost_tabu
    print(f"  CoÃ»t Tabu: {cost_tabu:.2f}")
    
    sol_genetic = memetic_algorithm(vrp, sol_tabu, pop_size=20, generations=80)
    cost_genetic = calculate_total_cost(sol_genetic, vrp.distances)
    results['Tabu + GÃ©nÃ©tique'] = cost_genetic
    print(f"  CoÃ»t final: {cost_genetic:.2f} (temps: {time.time()-start:.2f}s)\n")
    
    # Recuit SimulÃ© + GÃ©nÃ©tique + Tabu
    print("\nâ†’ SA + GÃ‰NÃ‰TIQUE + TABU (ULTRA)")
    start = time.time()
    sol_ultra1 = memetic_algorithm(vrp, sol_sa, pop_size=20, generations=80)
    sol_ultra2 = tabu_search_optimized(vrp, sol_ultra1, max_iter=800)
    cost_ultra = calculate_total_cost(sol_ultra2, vrp.distances)
    results['SA + GA + Tabu'] = cost_ultra
    print(f"  CoÃ»t final: {cost_ultra:.2f} (temps: {time.time()-start:.2f}s)\n")
        # VNS + Recuit SimulÃ© (ou inverse)
    print("\nâ†’ VNS + RECUIT SIMULÃ‰")
    start = time.time()
    sol_vns = variable_neighborhood_search(vrp, best_base, max_time=15.0)
    sol_vns_sa = simulated_annealing_ultra(vrp, sol_vns, max_time=15.0)
    cost_vns_sa = calculate_total_cost(sol_vns_sa, vrp.distances)
    results['VNS + SA'] = cost_vns_sa
    print(f"  CoÃ»t: {cost_vns_sa:.2f} (temps: {time.time()-start:.2f}s)\n")
    # ============================================================================
# PHASE B : HYBRIDATIONS DOUBLES
# ============================================================================

    print("\n" + "="*80)
    print("PHASE B: HYBRIDATIONS DOUBLES")
    print("="*80)

# GÃ©nÃ©tique â†’ Tabu
    print("\nâ†’ GÃ‰NÃ‰TIQUE â†’ TABU")
    start = time.time()
    sol_gen_tabu = genetic_then_tabu(vrp, best_base, max_time=60.0)
    cost_gen_tabu = calculate_total_cost(sol_gen_tabu, vrp.distances)
    results['GÃ©nÃ©tique â†’ Tabu'] = cost_gen_tabu
    print(f"  CoÃ»t: {cost_gen_tabu:.2f} (temps: {time.time()-start:.2f}s)\n")

# Tabu â†’ GÃ©nÃ©tique
    print("\nâ†’ TABU â†’ GÃ‰NÃ‰TIQUE")
    start = time.time()
    sol_tabu_gen = tabu_then_genetic(vrp, best_base, max_time=60.0)
    cost_tabu_gen = calculate_total_cost(sol_tabu_gen, vrp.distances)
    results['Tabu â†’ GÃ©nÃ©tique'] = cost_tabu_gen
    print(f"  CoÃ»t: {cost_tabu_gen:.2f} (temps: {time.time()-start:.2f}s)\n")

# GÃ©nÃ©tique â†’ VNS
    print("\nâ†’ GÃ‰NÃ‰TIQUE â†’ VNS")
    start = time.time()
    sol_gen_vns = genetic_then_vns(vrp, best_base, max_time=60.0)
    cost_gen_vns = calculate_total_cost(sol_gen_vns, vrp.distances)
    results['GÃ©nÃ©tique â†’ VNS'] = cost_gen_vns
    print(f"  CoÃ»t: {cost_gen_vns:.2f} (temps: {time.time()-start:.2f}s)\n")

# VNS â†’ GÃ©nÃ©tique
    print("\nâ†’ VNS â†’ GÃ‰NÃ‰TIQUE")
    start = time.time()
    sol_vns_gen = vns_then_genetic(vrp, best_base, max_time=60.0)
    cost_vns_gen = calculate_total_cost(sol_vns_gen, vrp.distances)
    results['VNS â†’ GÃ©nÃ©tique'] = cost_vns_gen
    print(f"  CoÃ»t: {cost_vns_gen:.2f} (temps: {time.time()-start:.2f}s)\n")


# ============================================================================
# PHASE C : HYBRIDATIONS TRIPLES
# ============================================================================

    print("\n" + "="*80)
    print("PHASE C: HYBRIDATIONS TRIPLES")
    print("="*80)

# GÃ©nÃ©tique â†’ Tabu â†’ VNS
    print("\nâ†’ GÃ‰NÃ‰TIQUE â†’ TABU â†’ VNS")
    start = time.time()
    sol_gen_tabu_vns = genetic_tabu_vns(vrp, best_base, max_time=90.0)
    cost_gen_tabu_vns = calculate_total_cost(sol_gen_tabu_vns, vrp.distances)
    results['GÃ©nÃ©tique â†’ Tabu â†’ VNS'] = cost_gen_tabu_vns
    print(f"  CoÃ»t: {cost_gen_tabu_vns:.2f} (temps: {time.time()-start:.2f}s)\n")

# Tabu â†’ GÃ©nÃ©tique â†’ Tabu
    print("\nâ†’ TABU â†’ GÃ‰NÃ‰TIQUE â†’ TABU")
    start = time.time()
    sol_tabu_gen_tabu = tabu_genetic_tabu(vrp, best_base, max_time=90.0)
    cost_tabu_gen_tabu = calculate_total_cost(sol_tabu_gen_tabu, vrp.distances)
    results['Tabu â†’ GÃ©nÃ©tique â†’ Tabu'] = cost_tabu_gen_tabu
    print(f"  CoÃ»t: {cost_tabu_gen_tabu:.2f} (temps: {time.time()-start:.2f}s)\n")

# VNS â†’ GÃ©nÃ©tique â†’ Tabu
    print("\nâ†’ VNS â†’ GÃ‰NÃ‰TIQUE â†’ TABU")
    start = time.time()
    sol_vns_gen_tabu = vns_genetic_tabu(vrp, best_base, max_time=90.0)
    cost_vns_gen_tabu = calculate_total_cost(sol_vns_gen_tabu, vrp.distances)
    results['VNS â†’ GÃ©nÃ©tique â†’ Tabu'] = cost_vns_gen_tabu
    print(f"  CoÃ»t: {cost_vns_gen_tabu:.2f} (temps: {time.time()-start:.2f}s)\n")


# ============================================================================
# PHASE D : HYBRIDATION ADAPTATIVE (CHAMPION) ðŸ†
# ============================================================================

    print("\n" + "="*80)
    print("PHASE D: HYBRIDATION ADAPTATIVE (CHAMPION) ðŸ†")
    print("="*80)

    print("\nâ†’ ðŸ† GÃ‰NÃ‰TIQUE â†” TABU ADAPTATIF")
    start = time.time()
    sol_champion = adaptive_genetic_tabu(vrp, best_base, max_time=120.0)
    cost_champion = calculate_total_cost(sol_champion, vrp.distances)
    results['ðŸ† GÃ©nÃ©tique â†” Tabu Adaptatif'] = cost_champion
    
    # 4. RÃ©sultats finaux
    print("\n" + "="*80)
    print("RÃ‰SULTATS FINAUX")
    print("="*80)
    
    sorted_results = sorted(results.items(), key=lambda x: x[1])
    for algo, cost in sorted_results:
        print(f"{algo:30s}: {cost:10.2f}")
    
    print(f"\n{'='*80}")
    print(f"MEILLEURE SOLUTION: {sorted_results[0][0]}")
    print(f"COÃ›T: {sorted_results[0][1]:.2f}")
    print(f"{'='*80}")
    
    # Afficher la meilleure solution
    best_algo = sorted_results[0][0]
    best_solutions_map = {
    'Tabu Search (VNS init)': sol_tabu,
    'GÃ©nÃ©tique Seul': sol_memetic,
    'GÃ©nÃ©tique â†’ Tabu': sol_gen_tabu,
    'Tabu â†’ GÃ©nÃ©tique': sol_tabu_gen,
    'GÃ©nÃ©tique â†’ VNS': sol_gen_vns,
    'VNS â†’ GÃ©nÃ©tique': sol_vns_gen,
    'GÃ©nÃ©tique â†’ Tabu â†’ VNS': sol_gen_tabu_vns,
    'Tabu â†’ GÃ©nÃ©tique â†’ Tabu': sol_tabu_gen_tabu,
    'VNS â†’ GÃ©nÃ©tique â†’ Tabu': sol_vns_gen_tabu,
    'ðŸ† GÃ©nÃ©tique â†” Tabu Adaptatif': sol_champion,
    }
    best_solution = best_solutions_map.get(best_algo, sol_champion)
    print("\nDÃ©tails de la meilleure solution:")
    for i, route in enumerate(best_solution, 1):
        load = sum(vrp.demands[c] for c in route)
        cost_route = calculate_route_cost(route, vrp.distances)
        print(f"  Route {i}: {len(route)} clients, charge={load}/{vrp.capacity}, coÃ»t={cost_route:.2f}")
        print(f"    Parcours: 0 â†’ {' â†’ '.join(map(str, route))} â†’ 0")

if __name__ == "__main__":
    main()

    
   
    

SYSTÃˆME VRP ULTRA-OPTIMISÃ‰ - F-n135-k7
  DIMENSION trouvée: 135
  CAPACITY trouvée: 2210
  Lecture des coordonnées...
  Lecture des demandes...
  Coordonnées chargées: 135 nœuds
  Demandes chargées: 135 nœuds
  Nombre de véhicules estimé: 7

✓ Fichier chargé avec succès!
  → 134 clients
  → 7 véhicules
  → Capacité: 2210

Dataset: 134 clients, 7 vÃ©hicules, capacitÃ©=2210

PHASE 1: ALGORITHMES DE BASE
  → Nearest Neighbor Amélioré...
  CoÃ»t: 1794.61 (temps: 0.29s)

  → Clarke-Wright Amélioré...
  CoÃ»t: 1207.17 (temps: 0.17s)

  → Sweep Algorithm...
  CoÃ»t: 2768.10 (temps: 0.00s)


PHASE 2: RECUIT SIMULÃ‰ ULTRA-OPTIMISÃ‰
  → Recuit Simulé Ultra-Optimisé...
    Nouvelle meilleure solution: 1203.68
    Nouvelle meilleure solution: 1199.03
    Nouvelle meilleure solution: 1193.45
    Nouvelle meilleure solution: 1190.96
    Nouvelle meilleure solution: 1185.37
    Nouvelle meilleure solution: 1183.16
    Temps écoulé: 40.1s, T=964.61, Meilleur=1183.16
  âš ï¸ Recuit SimulÃ© â‰¥1183 (